# Deploy and Run the Executor Function

This interactive guide shows how to upload the executor function to Qiskit Serverless and run its two modes. The executor is, morally, a raw sampler: `mode="sampler"` returns raw bit-array results, while `mode="estimator"` wraps that same sampler to reconstruct expectation values. Two examples follow: a multi-observable survey in estimator mode with a sampler cross-check on Z-diagonal observables, and an error-mitigation comparison that exercises dynamical decoupling versus Pauli twirling.

### Requirements

This guide was developed with the following local package versions:

In [ ]:
from qiskit import __version__ as qiskit_version
from qiskit_ibm_catalog import __version__ as catalog_version
import numpy as np

print("qiskit version:", qiskit_version)
print("qiskit-ibm-catalog version:", catalog_version)
print("numpy version:", np.__version__)

## 1. Authentication

Use `qiskit-ibm-catalog` to authenticate to `QiskitServerless` with your API key (token) and CRN (instance), which you can find on the [IBM Quantum Platform](https://quantum.cloud.ibm.com) dashboard. This will allow you to locally instantiate the serverless client to upload or run the selected function:

```python
from qiskit_ibm_catalog import QiskitServerless
serverless = QiskitServerless(channel="ibm_quantum_platform", token="MY_TOKEN", instance="MY_CRN")
```

You can optionally use `save_account()` to save your credentials in your local environment (see the [Set up your IBM Cloud account](/docs/guides/cloud-setup#cloud-save) guide).

In [ ]:
from qiskit_ibm_catalog import QiskitServerless

# Authenticate to the remote cluster
# In this case, loading a named saved account
# serverless = QiskitServerless(name="my_account")

# REPLACE WITH YOUR OWN CREDENTIALS or SAVED ACCOUNT
serverless = QiskitServerless(channel="ibm_quantum_platform", token="MY_TOKEN", instance="MY_CRN")

## 2. Upload the custom function

To upload a Qiskit Function, you must first instantiate a `QiskitFunction` object that defines the function source code. The title will allow you to identify the function once it's in the remote cluster. The main entry point is the file that contains `if __name__ == "__main__"`. The `working_dir` contains the entrypoint together with the `options/` sub-package it imports at runtime.

In [ ]:
from qiskit_ibm_catalog import QiskitFunction

template = QiskitFunction(
    title="executor_function_template",
    entrypoint="executor_entrypoint.py",
    working_dir="./source_files/",  # all files in this directory will be uploaded
)
print(template)

Once the instance is ready, upload it to serverless:

In [ ]:
serverless.upload(template)

To check if the program successfully uploaded, use `serverless.list()`:

In [ ]:
serverless.list()

## 3. Load and run the custom function remotely

The function template has been uploaded, so you can run it remotely with Qiskit Serverless. First, load the template by name:

In [ ]:
template = serverless.load("executor_function_template")
print(template)

### Example 1 — Multi-observable survey, and a sampler cross-check

This example measures five observables on a fixed random 3-qubit circuit. In `mode="estimator"` each observable is its own PUB, so one `run()` returns five `PubResult`s with an expectation value each. We then run the **same circuit** in `mode="sampler"` and reconstruct the Z-diagonal observables from the computational-basis counts. A sampler only ever measures in the Z basis, so it reproduces `ZZI`/`IZZ`/`ZIZ` directly but cannot see `XXI`/`IYY` — recovering those off-diagonal observables is exactly what the basis rotations in estimator mode provide.

In [ ]:
import numpy as np
from qiskit.circuit.random import random_circuit

NUM_QUBITS = 3

# A fixed random circuit (no measurements — estimator mode appends its own).
circuit = random_circuit(num_qubits=NUM_QUBITS, depth=5, seed=40)

# Z-only strings are diagonal in the computational basis; XX/YY are not.
obs_labels = ["ZZI", "IZZ", "ZIZ", "XXI", "IYY"]

print("num_qubits:", NUM_QUBITS)
print("observables:", obs_labels)

Run all observables in **estimator** mode — one PUB per observable, so a single call returns five `PubResult`s:

In [ ]:
job_estimator = template.run(
    backend_name="ibm_kingston",
    pubs=[(circuit, obs) for obs in obs_labels],
    mode="estimator",
    options={"default_precision": 0.02},
)
print("job_id:", job_estimator.job_id)

Poll until the job reaches a terminal state:

In [ ]:
import time

last_status = None
while True:
    status = job_estimator.status()
    if status != last_status:
        print(f"job_estimator: {status}")
        last_status = status
    if status in ("DONE", "ERROR", "CANCELED"):
        break
    time.sleep(5)

Retrieve the expectation value of each observable (each `data.evs` is a scalar here):

In [ ]:
result_estimator = job_estimator.result()
evs_estimator = {
    label: float(pr.data.evs) for label, pr in zip(obs_labels, result_estimator["hw_results"])
}
for label, val in evs_estimator.items():
    print(f"{label}: {val:+.4f}")

Now run the **same circuit** in `sampler` mode. We append measurements and submit a single sampler PUB; the result is a raw bit array in the Z basis:

In [ ]:
sampler_circuit = circuit.copy()
sampler_circuit.measure_all()

job_sampler = template.run(
    backend_name="ibm_kingston",
    pubs=[(sampler_circuit,)],
    mode="sampler",
    options={"default_shots": 4096},
)
print("job_id:", job_sampler.job_id)

Poll until the sampler job finishes:

In [ ]:
last_status = None
while True:
    status = job_sampler.status()
    if status != last_status:
        print(f"job_sampler: {status}")
        last_status = status
    if status in ("DONE", "ERROR", "CANCELED"):
        break
    time.sleep(5)

Reconstruct the Z-diagonal observables from the sampler bit array (via `BitArray.expectation_values`) and compare against estimator mode. They should agree within shot noise. The off-diagonal observables are shown as estimator-only — a Z-basis sampler cannot produce them:

In [ ]:
from qiskit.quantum_info import Pauli

result_sampler = job_sampler.result()
counts = result_sampler["hw_results"][0].data.meas  # BitArray in the Z basis

z_diagonal = ["ZZI", "IZZ", "ZIZ"]
print(f"{'Observable':12s}{'estimator':>12s}{'sampler':>12s}{'delta':>10s}")
print("-" * 46)
for label in z_diagonal:
    est = evs_estimator[label]
    smp = float(counts.expectation_values(Pauli(label)))
    print(f"{label:12s}{est:>+12.4f}{smp:>+12.4f}{est - smp:>+10.4f}")

print("\nOff-diagonal observables (estimator only):")
for label in ["XXI", "IYY"]:
    print(f"  {label}: {evs_estimator[label]:+.4f}")

### Example 2 — Error mitigation comparison

Real hardware introduces decoherence, gate noise, and measurement crosstalk. The executor exposes a structured `options` dict with a `mitigation_level` shortcut. This example fixes a single parameter set on a 4-qubit `RealAmplitudes` ansatz so the two strategies are directly comparable, and runs both in estimator mode.

| Job | `mitigation_level` | What this exercises |
|---|---|---|
| `job_l1` | `1` (default) | Dynamical decoupling only |
| `job_l2` | `2` | DD (`XY4` sequence) + gate & measurement twirling |

`mitigation_level=2` enables Pauli twirling, which routes the program through the executor's samplex (boxed-layer) path — so this comparison only runs on real hardware.

In [ ]:
import numpy as np
from qiskit.circuit.library import real_amplitudes
from qiskit.quantum_info import SparsePauliOp

NUM_QUBITS = 4

ansatz = real_amplitudes(num_qubits=NUM_QUBITS, reps=2)  # 12 parameters
observable = SparsePauliOp("IIZZ")  # ZZ correlation on qubits 0, 1

# Fix a single parameter set so the two jobs are directly comparable.
rng = np.random.default_rng(seed=7)
params = rng.uniform(0, 2 * np.pi, size=(ansatz.num_parameters,))

# Level 1 — dynamical decoupling only (default).
options_l1 = {"mitigation_level": 1, "default_precision": 0.02}

# Level 2 — adds gate + measurement twirling and switches DD to the XY4 sequence.
options_l2 = {
    "mitigation_level": 2,
    "default_precision": 0.02,
    "dynamical_decoupling": {"sequence_type": "XY4"},
}

Submit both jobs. Launching them before either has finished means both queue and execute in parallel on the backend:

In [ ]:
BACKEND = "ibm_kingston"

job_l1 = template.run(
    backend_name=BACKEND,
    pubs=[(ansatz, observable, params)],
    mode="estimator",
    options=options_l1,
)
job_l2 = template.run(
    backend_name=BACKEND,
    pubs=[(ansatz, observable, params)],
    mode="estimator",
    options=options_l2,
)

print("job_l1 id:", job_l1.job_id)
print("job_l2 id:", job_l2.job_id)

Poll both jobs to completion. Each is polled in a round-robin loop so neither blocks the other:

In [ ]:
import time

terminal = {"DONE", "ERROR", "CANCELED"}
last_s1, last_s2 = None, None

while True:
    s1 = job_l1.status()
    s2 = job_l2.status()
    if s1 != last_s1 or s2 != last_s2:
        print(f"job_l1={s1}  job_l2={s2}")
        last_s1, last_s2 = s1, s2
    if s1 in terminal and s2 in terminal:
        break
    time.sleep(2)

Retrieve results from both jobs and compare the expectation value and standard deviation side by side. A lower `std` at a comparable `evs` indicates the higher mitigation level is recovering more signal from the noisy backend without biasing the estimate:

In [ ]:
result_l1 = job_l1.result()
result_l2 = job_l2.result()

pr_l1 = result_l1["hw_results"][0]
pr_l2 = result_l2["hw_results"][0]

evs_l1 = float(pr_l1.data.evs)
stds_l1 = float(pr_l1.data.stds)
evs_l2 = float(pr_l2.data.evs)
stds_l2 = float(pr_l2.data.stds)

print("Observable : IIZZ")
print("=" * 48)
print(f"{'':14s} {'evs':>10s}  {'std':>10s}")
print(f"{'Level 1':14s} {evs_l1:+.4f}  {stds_l1:10.4f}")
print(f"{'Level 2':14s} {evs_l2:+.4f}  {stds_l2:10.4f}")
print("=" * 48)
delta_evs = evs_l2 - evs_l1
delta_stds = stds_l2 - stds_l1
direction = "lower std with level 2" if delta_stds < 0 else "higher std with level 2"
print(f"delta evs (L2 - L1) :  {delta_evs:+.4f}")
print(f"delta std (L2 - L1) :  {delta_stds:+.4f}  ({direction})")

You can also retrieve the resource metadata for each job. Level 2 runs more circuit variants (twirling randomizations), so `EXECUTING_QPU` time will be proportionally higher:

In [ ]:
stages = [
    "RUNNING: OPTIMIZING_FOR_HARDWARE",
    "RUNNING: WAITING_FOR_QPU",
    "RUNNING: EXECUTING_QPU",
    "RUNNING: POST_PROCESSING",
]

meta_l1 = result_l1["metadata"]["resources_usage"]
meta_l2 = result_l2["metadata"]["resources_usage"]

print(f"{'Stage':<38s} {'Level 1 (s)':>12s}  {'Level 2 (s)':>12s}")
print("-" * 60)
for stage in stages:
    t1 = meta_l1.get(stage, {}).get("CPU_TIME", float("nan"))
    t2 = meta_l2.get(stage, {}).get("CPU_TIME", float("nan"))
    print(f"{stage:<38s} {t1:>12.4f}  {t2:>12.4f}")